# 5.16 配比怎麼落到實際 batch？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一次更新的生命週期**

忘記清零會累加；累積梯度時才刻意保留。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/backward.svg")))

**先想一想：**90筆每筆1token，加10筆每筆20token，主要在學誰？

設定90:10只是意圖；實際batch的來源數與有效答案tokens纔是執行結果。長答案來源即使筆數少，也可能佔大部分監督。

**把直覺寫成數學：**token配比=各來源有效token數/總有效token數，通常不等於樣本配比。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
counts = {"短答": 90, "長答": 10}
length = {"短答": 1, "長答": 20}
tokens = {k: counts[k] * length[k] for k in counts}
print(tokens, {k: v / sum(tokens.values()) for k, v in tokens.items()})

**如何讀結果：**混合資料時保留source標記，分項評估，固定共同token預算。

**只改一件事：**只把長答長度改爲1。
